In [1]:
# Analysis A: what the ATFS-1 transcriptional output is actually made of
# (Day 6, roadmap Part 3).
#
# Analysis B answered one pre-specified question with a hand-built census: how many
# of the 61 regulon genes are folding machinery? (2, strict.) This asks the open
# version - across every annotated biological category, which ones are over- or
# under-represented among ATFS-1 targets relative to the genes that could have been
# detected in the same experiment?
#
# Two choices decide whether that question is answerable rather than decorative:
# which background the counts are compared against, and which gene list gets called
# "the targets". Both are made explicit here, and both are varied at the end.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd
import numpy as np
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

# The ontology itself. GO terms are nested - "protein refolding" sits underneath
# "protein folding" - so a gene annotated only to the specific child still has to
# count toward the parent, or every general category comes out undercounted. That
# needs the real term graph, not just the annotation table.
OBO = "ref_data/go/go-basic.obo"
terms, alt_id_map = {}, {}
cur = None

def _store(term):
    if term and term["id"]:
        terms[term["id"]] = term

with open(OBO) as fh:
    for line in fh:
        line = line.rstrip("\n")
        if line.startswith("["):
            _store(cur)
            cur = {"id": None, "name": None, "ns": None, "parents": set(),
                   "obsolete": False} if line == "[Term]" else None
            continue
        if cur is None or not line:
            continue
        key, _, val = line.partition(": ")
        if key == "id":
            cur["id"] = val
        elif key == "name":
            cur["name"] = val
        elif key == "namespace":
            cur["ns"] = val
        elif key == "alt_id":
            alt_id_map[val] = cur["id"]
        elif key == "is_obsolete" and val == "true":
            cur["obsolete"] = True
        elif key == "is_a":
            cur["parents"].add(val.split(" ! ")[0].strip())
        elif key == "relationship" and val.startswith("part_of "):
            cur["parents"].add(val.split()[1])
_store(cur)

# is_a and part_of are the two relations GO annotations propagate over. The others
# (regulates, occurs_in, ...) deliberately do not: a gene that regulates folding is
# not thereby a folding gene.
_ancestor_cache = {}
def ancestors(term_id):
    if term_id in _ancestor_cache:
        return _ancestor_cache[term_id]
    _ancestor_cache[term_id] = set()          # guards against cycles mid-recursion
    found = set()
    for parent in terms.get(term_id, {}).get("parents", ()):
        if parent in terms:
            found.add(parent)
            found |= ancestors(parent)
    _ancestor_cache[term_id] = found
    return found

# Structural checks against known ontology facts, before anything depends on this.
assert terms["GO:0006457"]["name"] == "protein folding"
assert "GO:0006457" in ancestors("GO:0042026"), "refolding must sit under protein folding"
assert "GO:0008150" in ancestors("GO:0006457"), "every process term must reach the root"

print(f"GO ontology: {len(terms):,} terms "
      f"({sum(not t['obsolete'] for t in terms.values()):,} current), "
      f"{len(alt_id_map):,} secondary IDs")

GO ontology: 48,340 terms (38,092 current), 3,646 secondary IDs


In [2]:
# The annotations. Two classes of row are dropped. NOT-qualified annotations assert
# that a gene does *not* have a function, and would otherwise be counted as if it
# did. ND ("no biological data available") is a placeholder recording that nothing
# is known about the gene - counting it as evidence would turn ignorance into a
# finding.
GAF = "ref_data/go/wb.gaf.gz"
records = []
with gzip.open(GAF, "rt") as fh:
    for line in fh:
        if line.startswith("!"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 15 or f[12] != "taxon:6239":
            continue
        records.append((f[1], f[3], f[4], f[6], f[8]))

gaf = pd.DataFrame(records, columns=["gid", "qualifier", "go_id", "evidence", "aspect"])
n_not = gaf["qualifier"].str.startswith("NOT").sum()
n_nd = (gaf["evidence"] == "ND").sum()
gaf = gaf[~gaf["qualifier"].str.startswith("NOT") & (gaf["evidence"] != "ND")]
gaf["go_id"] = gaf["go_id"].map(lambda t: alt_id_map.get(t, t))

# Propagate every annotation up to all of its ancestor terms.
gene_terms = {}
for gid, go_id in zip(gaf["gid"], gaf["go_id"]):
    if go_id not in terms or terms[go_id]["obsolete"]:
        continue
    gene_terms.setdefault(gid, set()).add(go_id)
    gene_terms[gid] |= ancestors(go_id)

term_genes = {}
for gid, tset in gene_terms.items():
    for term in tset:
        term_genes.setdefault(term, set()).add(gid)

print(f"GAF: {len(gaf):,} usable annotations over {len(gene_terms):,} genes "
      f"(dropped {n_not} NOT-qualified, {n_nd} ND)")
print(f"After ancestor propagation: {sum(len(v) for v in gene_terms.values()):,} "
      f"gene-term pairs across {len(term_genes):,} terms")

# hsp-6 is the field's standard UPRmt reporter and an unambiguous chaperone. If
# propagation works it must now carry the general "protein folding" term, which it
# is not directly annotated to in every case.
HSP6 = "WBGene00002010"
assert "GO:0006457" in gene_terms[HSP6], "hsp-6 should reach protein folding"
print(f"Check: hsp-6 carries {len(gene_terms[HSP6])} terms after propagation, "
      f"protein folding among them")

GAF: 103,606 usable annotations over 12,282 genes (dropped 64 NOT-qualified, 182 ND)
After ancestor propagation: 471,587 gene-term pairs across 9,383 terms
Check: hsp-6 carries 63 terms after propagation, protein folding among them


In [3]:
# The background - the choice the roadmap flags as the single easiest thing for a
# reviewer to reject, and not a formality. Comparing a gene list against all ~20,000
# genes in the genome silently includes thousands that were never eligible to appear
# in it: sperm-specific, embryo-specific, or simply not transcribed under these
# conditions. Any category that happens to be broadly expressed then looks
# "enriched" in almost any real experimental list, for reasons that have nothing to
# do with the experiment.
#
# The regulon and both Wu 2018 lists all come from GSE110984, so the honest
# comparison pool is the genes that survived that experiment's own expression
# filter - the authors' threshold, not one invented here. The genome-wide
# alternative is run at the end to show what the choice is worth.
CPM = "ref_data/GSE110984/GSE110984_normalized_filtered_CPM_table_VANJ_20171130.txt.gz"
expressed = set(pd.read_csv(CPM, sep="\t")["ens_gene"].dropna())

ids = pd.read_csv("ref_data/c_elegans.PRJNA13758.WS285.geneIDs.txt.gz", header=None,
                  names=["taxon", "gid", "public_name", "seqname", "status", "biotype"])
seq_to_gid = {str(s).lower(): g for s, g in zip(ids["seqname"], ids["gid"]) if pd.notna(s)}
gid_to_name = {g: (n if pd.notna(n) else s)
               for g, n, s in zip(ids["gid"], ids["public_name"], ids["seqname"])}

background = expressed & set(gene_terms)
unannotated_bg = len(expressed) - len(background)
print(f"Expressed genes (GSE110984 filter): {len(expressed):,}")
print(f"...carrying at least one GO annotation: {len(background):,} "
      f"({100*len(background)/len(expressed):.1f}%)")
print(f"\n{unannotated_bg:,} expressed genes have no annotation at all and cannot enter")
print("an enrichment test in either direction. They are counted separately below.")

Expressed genes (GSE110984 filter): 13,837
...carrying at least one GO annotation: 9,371 (67.7%)

4,466 expressed genes have no annotation at all and cannot enter
an enrichment test in either direction. They are counted separately below.


In [4]:
# The gene lists, and why there are four of them.
#
# Soo's 61 is an intersection of three separate induction experiments: a gene has to
# clear significance in nuo-6, in atfs-1(et15) and in atfs-1(et17) at once. That is
# a strict AND, and a strict AND removes genes for statistical reasons as well as
# biological ones - hsp-6 itself, the field's own reporter for this pathway, is
# induced in two of the three and is not on the list. So "folding machinery is rare
# among the 61" could in part be a property of the filter rather than of ATFS-1.
#
# The way to find out is to walk the same question back up the filtering series to
# the looser lists the 61 was carved out of - same dataset, same background, fewer
# simultaneous conditions required.
soo = pd.read_excel("data/raw/ATFS1_targets_Soo.xlsx", sheet_name="Sheet1")
soo = soo.iloc[0:64].dropna(subset=["Gene name"]).rename(
    columns={"Gene sequence \nname": "seqname"})
regulon_df = soo[~soo["Gene name"].isin(["hsp-6", "hsp-60"])]
regulon = {seq_to_gid[str(s).lower()] for s in regulon_df["seqname"]}
if len(regulon) != 61:
    raise RuntimeError(f"Expected 61 regulon genes, resolved {len(regulon)}.")

WU = "data/raw/wu2018_AdditionalFile2.xlsx"
sheet = lambda name: pd.read_excel(WU, sheet_name=name, header=2)
up = lambda df: set(df[df["logFC"] > 0]["ens_gene"])
nuo6_dependent = up(sheet("nuo-6")) - up(sheet("nuo-6;atfs-1"))
et_both = up(sheet("atfs-1(et15)")) & up(sheet("atfs-1(et17)"))
three_way = nuo6_dependent & et_both

GENE_SETS = {
    "nuo-6 dependent (1 condition)": nuo6_dependent,
    "et15 & et17 (2 conditions)": et_both,
    "all three conditions": three_way,
    "Soo high-confidence 61": regulon,
}
for label, genes in GENE_SETS.items():
    in_expr = genes & expressed
    print(f"{label:32s} n={len(genes):5d}  annotated={len(genes & background):5d}  "
          f"unannotated={len(in_expr) - len(genes & background):4d}")

if not regulon <= three_way:
    raise RuntimeError("The 61 should be a subset of the three-way intersection.")
print(f"\nNesting confirmed: the 61 sit inside the {len(three_way)}-gene three-way "
      f"intersection,\nwhich sits inside both parent lists. Note the three-way "
      f"intersection is much larger\nthan 61 - Soo applied further filtering beyond "
      f"the intersection itself.")

nuo-6 dependent (1 condition)    n= 1673  annotated= 1085  unannotated= 588
et15 & et17 (2 conditions)       n=  529  annotated=  262  unannotated= 267
all three conditions             n=  231  annotated=  121  unannotated= 110
Soo high-confidence 61           n=   61  annotated=   42  unannotated=  19

Nesting confirmed: the 61 sit inside the 231-gene three-way intersection,
which sits inside both parent lists. Note the three-way intersection is much larger
than 61 - Soo applied further filtering beyond the intersection itself.


In [5]:
# The test. For each GO term: of the background genes, K carry the term; of the
# query genes, k do. The hypergeometric distribution gives the probability of seeing
# at least k (enrichment) or at most k (depletion) if the query were a random draw
# from the background. Both tails are computed - here the question is as much about
# what is missing as about what is present.
#
# Terms are only tested where they have between 5 and 1,000 background genes. Below
# that, no result can survive correction; above it, the term is too generic to mean
# anything. p-values are corrected across all tested terms with Benjamini-Hochberg.
MIN_TERM, MAX_TERM = 5, 1000
# Ties are broken on go_id so the table is byte-identical between runs. Terms
# with equal p carry identical counts, so this fixes display order only - it
# never changes which term is reported or any value in the row. Without it,
# set iteration order varies with Python's per-process hash seed and tied rows
# shuffle, which showed up as a spurious diff when the notebook was re-run.

def enrichment(query, bg):
    q = set(query) & bg
    rows = []
    for term, annotated in term_genes.items():
        K = len(annotated & bg)
        if not (MIN_TERM <= K <= MAX_TERM):
            continue
        k = len(annotated & q)
        rows.append((term, terms[term]["name"], terms[term]["ns"], k, len(q), K, len(bg),
                     hypergeom.sf(k - 1, len(bg), K, len(q)),
                     hypergeom.cdf(k, len(bg), K, len(q))))
    out = pd.DataFrame(rows, columns=["go_id", "term", "namespace", "k", "n", "K", "N",
                                      "p_enrich", "p_deplete"])
    for tail in ("enrich", "deplete"):
        out[f"fdr_{tail}"] = multipletests(out[f"p_{tail}"], method="fdr_bh")[1]
    out["expected"] = out["K"] * out["n"] / out["N"]
    out["fold"] = out["k"] / out["expected"].replace(0, np.nan)
    return out.sort_values(["p_enrich", "go_id"])

# Positive control. The chaperone/protease census was built independently, from Pfam
# protein domains, with no reference to GO at any point. If this pipeline cannot
# recover "protein folding" as enriched in a set of genes selected for being
# chaperones, it is broken, and nothing below it can be believed.
census = pd.read_csv("data/chaperone_protease_census.csv")
census_gids = {seq_to_gid[str(s).lower()] for s in census["seqname"]
               if str(s).lower() in seq_to_gid}
control = enrichment(census_gids, background)

print(f"--- Positive control: {len(census_gids & background)} census genes vs. background ---")
print(control.head(8)[["term", "namespace", "k", "expected", "fold", "fdr_enrich"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3g}"))

folding = control[control["go_id"] == "GO:0006457"].iloc[0]
if folding["fdr_enrich"] > 0.01:
    raise RuntimeError("Pipeline failed its positive control - protein folding not recovered.")
print(f"\nprotein folding in the census: {folding['k']} of {folding['n']} genes, "
      f"expected {folding['expected']:.1f} "
      f"({folding['fold']:.0f}x), FDR = {folding['fdr_enrich']:.2e}.")
print("Pipeline validated.")

--- Positive control: 62 census genes vs. background ---
                                   term          namespace  k  expected  fold  fdr_enrich
                        protein folding biological_process 45     0.794  56.7    1.55e-71
                     protein maturation biological_process 49      1.83  26.7    1.86e-61
                      protein refolding biological_process 22     0.159   139    6.46e-45
              protein folding chaperone molecular_function 19     0.179   106    4.19e-34
                       response to heat biological_process 20     0.463  43.2    9.39e-26
       response to temperature stimulus biological_process 20     0.629  31.8    7.43e-23
             heat shock protein binding molecular_function 15     0.212  70.8    1.04e-22
ATP-dependent protein folding chaperone molecular_function 11    0.0794   139    2.67e-21

protein folding in the census: 45 of 62 genes, expected 0.8 (57x), FDR = 1.55e-71.
Pipeline validated.


In [6]:
# Analysis A proper: the 61-gene regulon against the expressed background.
regulon_go = enrichment(regulon, background)

n_annotated = len(regulon & background)
rate_regulon = 100 * (61 - n_annotated) / 61
rate_background = 100 * unannotated_bg / len(expressed)
print(f"Of the 61 regulon genes, {n_annotated} carry a GO annotation and "
      f"{61 - n_annotated} ({rate_regulon:.0f}%) carry none.")
print(f"Among expressed genes generally, {rate_background:.0f}% carry none - so on this")
print("measure the regulon is no less characterised than the transcriptome it came from.")
print("That is worth stating plainly because it is easy to assume otherwise: 45.9% of")
print("the 61 carry no gene *symbol*, but a gene can lack a name and still carry")
print("inferred annotation, and here the two rates are not the same number.")
print(f"\nEither way those {61 - n_annotated} genes cannot enter the test below in either")
print("direction, so every count that follows is out of "
      f"{n_annotated}, not 61.")

print(f"\n--- Enriched (FDR < 0.05) ---")
sig = regulon_go[regulon_go["fdr_enrich"] < 0.05]
cols = ["term", "namespace", "k", "expected", "fold", "p_enrich", "fdr_enrich"]
if len(sig) == 0:
    print("Nothing reaches FDR < 0.05.\n")
    print("Strongest uncorrected signals, for context only - these are NOT results:")
    print(regulon_go.head(10)[cols].to_string(index=False, float_format=lambda x: f"{x:.3g}"))
else:
    print(sig[cols].to_string(index=False, float_format=lambda x: f"{x:.3g}"))

print(f"\n--- Depleted (FDR < 0.05) ---")
dep = regulon_go[regulon_go["fdr_deplete"] < 0.05].sort_values(["p_deplete", "go_id"])
dcols = ["term", "namespace", "k", "expected", "fold", "p_deplete", "fdr_deplete"]
print("Nothing reaches FDR < 0.05." if len(dep) == 0 else
      dep[dcols].head(10).to_string(index=False, float_format=lambda x: f"{x:.3g}"))

Of the 61 regulon genes, 42 carry a GO annotation and 19 (31%) carry none.
Among expressed genes generally, 32% carry none - so on this
measure the regulon is no less characterised than the transcriptome it came from.
That is worth stating plainly because it is easy to assume otherwise: 45.9% of
the 61 carry no gene *symbol*, but a gene can lack a name and still carry
inferred annotation, and here the two rates are not the same number.

Either way those 19 genes cannot enter the test below in either
direction, so every count that follows is out of 42, not 61.

--- Enriched (FDR < 0.05) ---
                            term          namespace  k  expected  fold  p_enrich  fdr_enrich
glucuronosyltransferase activity molecular_function  5     0.291  17.2  9.58e-06      0.0411

--- Depleted (FDR < 0.05) ---
Nothing reaches FDR < 0.05.


In [7]:
# The intersection-artifact test. Track the folding categories across the filtering
# series: if chaperones are proportionally common in the loose lists and vanish only
# in the strict one, the shortage is being manufactured by the AND filter. If they
# are equally scarce everywhere, it is a real property of the ATFS-1 response.
# "Folding-related" is defined structurally from the ontology rather than by picking
# terms by hand: three current roots and everything underneath them. Hand-picking is
# how the two obsolete terms an earlier version of this cell used - "unfolded protein
# binding" (GO:0051082) and "chaperone-mediated protein folding" (GO:0061077) -
# silently contributed zero-gene rows that looked like real absences of signal.
FOLDING_ROOTS = ["GO:0006457", "GO:0044183", "GO:0140309"]

children = {}
for term, meta in terms.items():
    for parent in meta["parents"]:
        children.setdefault(parent, set()).add(term)

def descendants(root):
    out, stack = set(), [root]
    while stack:
        for child in children.get(stack.pop(), ()):
            if child not in out:
                out.add(child)
                stack.append(child)
    return out

for root in FOLDING_ROOTS:
    if terms[root]["obsolete"]:
        raise RuntimeError(f"{root} is obsolete - do not build a category on it.")
folding_terms = set(FOLDING_ROOTS) | {d for r in FOLDING_ROOTS for d in descendants(r)}
folding_terms = {t for t in folding_terms if not terms[t]["obsolete"]}
folding_genes = set().union(*(term_genes.get(t, set()) for t in folding_terms))
print(f"Folding-related category: {len(folding_terms)} current GO terms "
      f"({', '.join(terms[r]['name'] for r in FOLDING_ROOTS)} and descendants),")
print(f"covering {len(folding_genes & background)} of the {len(background):,} background genes.\n")

def walk_category(category_genes, universe, sets, label_width=32):
    out = []
    K = len(category_genes & universe)
    for label, genes in sets.items():
        q = genes & universe
        k = len(category_genes & q)
        exp = K * len(q) / len(universe)
        out.append({"gene set": label, "tested": len(q), "obs": k, "exp": round(exp, 2),
                    "pct": round(100 * k / len(q), 2),
                    "fold": round(k / exp, 2) if exp else np.nan,
                    "p": hypergeom.sf(k - 1, len(universe), K, len(q))})
    return pd.DataFrame(out)

go_walk = walk_category(folding_genes, background, GENE_SETS)
print("--- Folding-related GO terms across the filtering series ---")
print(go_walk.to_string(index=False, float_format=lambda x: f"{x:.3g}"))

# The same walk using the hand-built census, which is not tied to GO at all - so a
# gap in GO annotation cannot produce or hide the pattern. Note this runs against
# every expressed gene, not only annotated ones, since census membership was decided
# from Pfam domains and does not require a GO record.
print("\n--- Chaperone/protease census membership across the same series ---")
census_walk = walk_category(census_gids, expressed, GENE_SETS)
print(census_walk.to_string(index=False, float_format=lambda x: f"{x:.3g}"))

# Read the direction off the numbers rather than asserting one.
for name, walk in [("GO folding-related", go_walk), ("Pfam census", census_walk)]:
    pcts = walk["pct"].tolist()
    rising = all(a <= b for a, b in zip(pcts, pcts[1:]))
    falling = all(a >= b for a, b in zip(pcts, pcts[1:]))
    trend = "rises" if rising else "falls" if falling else "does not move monotonically"
    print(f"\n{name}: {' -> '.join(f'{p:.2f}%' for p in pcts)} as the filter tightens ({trend}).")

last = census_walk.iloc[-1]
n_sets = len(GENE_SETS)
print(f"""
The artifact hypothesis predicted the opposite of this. If the three-condition AND
were manufacturing the shortage of folding machinery, representation would fall as
the filter tightened. It does not - so the scarcity of chaperones among the 61 is a
property of the ATFS-1 response, not of how Soo's list was assembled.

That resolves the artifact question and raises a different one. Against the
expressed background the 61 are not depleted of folding machinery at all: {int(last['obs'])} of
{int(last['tested'])} is {last['fold']:.1f}x the {last['exp']:.2f} expected by chance (p = {last['p']:.3g}).
Caveat before that gets quoted: p is uncorrected across the {n_sets} sets walked here
(Bonferroni: {min(1.0, last['p'] * n_sets):.3g}), and at {int(last['obs'])} observed genes one gene either way
moves it. It is a direction, not a significant finding - but it is emphatically not
evidence of under-representation, and the paper cannot claim depletion.""")

Folding-related category: 9 current GO terms (protein folding, protein folding chaperone, unfolded protein holdase activity and descendants),
covering 120 of the 9,371 background genes.

--- Folding-related GO terms across the filtering series ---
                     gene set  tested  obs  exp  pct  fold     p
nuo-6 dependent (1 condition)    1085    8 13.9 0.74  0.58 0.974
   et15 & et17 (2 conditions)     262    4 3.36 1.53  1.19 0.433
         all three conditions     121    3 1.55 2.48  1.94 0.202
       Soo high-confidence 61      42    1 0.54 2.38  1.86 0.419

--- Chaperone/protease census membership across the same series ---
                     gene set  tested  obs  exp  pct  fold      p
nuo-6 dependent (1 condition)    1673   12  8.1 0.72  1.48  0.105
   et15 & et17 (2 conditions)     529    4 2.56 0.76  1.56  0.253
         all three conditions     231    3 1.12  1.3  2.68  0.101
       Soo high-confidence 61      61    2  0.3 3.28  6.77 0.0352

GO folding-related: 0.74% -

In [8]:
# What the background choice is actually worth. Same regulon, same test, two
# comparison pools: the expressed set used above, and every annotated gene in the
# genome regardless of whether it could have been detected here.
genome_bg = set(gene_terms)
print(f"expressed background: {len(background):,} genes | "
      f"genome-wide background: {len(genome_bg):,} genes\n")

regulon_genome = enrichment(regulon, genome_bg)
comp = regulon_go.merge(regulon_genome, on="go_id", suffixes=("_expr", "_gnm"))
comp = comp.sort_values(["fdr_enrich_gnm", "go_id"]).head(12)[
    ["term_expr", "k_expr", "expected_expr", "fold_expr", "fdr_enrich_expr",
     "expected_gnm", "fold_gnm", "fdr_enrich_gnm"]]
comp.columns = ["term", "k", "exp(expr)", "fold(expr)", "FDR(expr)",
                "exp(genome)", "fold(genome)", "FDR(genome)"]
print("--- Strongest genome-wide hits, beside the same terms on the expressed background ---")
print(comp.to_string(index=False, float_format=lambda x: f"{x:.3g}"))

n_expr = int((regulon_go["fdr_enrich"] < 0.05).sum())
n_gnm = int((regulon_genome["fdr_enrich"] < 0.05).sum())
print(f"\nTerms reaching FDR < 0.05: {n_expr} on the expressed background, "
      f"{n_gnm} genome-wide.")

expressed background: 9,371 genes | genome-wide background: 12,282 genes



--- Strongest genome-wide hits, beside the same terms on the expressed background ---
                                  term  k  exp(expr)  fold(expr)  FDR(expr)  exp(genome)  fold(genome)  FDR(genome)
      glucuronosyltransferase activity  5      0.291        17.2     0.0411        0.284          17.6       0.0386
      UDP-glycosyltransferase activity  5      0.515         9.7      0.216        0.489          10.2         0.18
          transition metal ion binding 10       2.52        3.97      0.216         2.47          4.05         0.18
                         mitochondrion 10       3.49        2.86      0.571         2.78           3.6        0.353
          ubiquinone metabolic process  2     0.0493        40.6       0.41        0.041          48.7        0.363
       ubiquinone biosynthetic process  2     0.0493        40.6       0.41        0.041          48.7        0.363
           ketone biosynthetic process  2     0.0493        40.6       0.41        0.041          48.7